# Zynthe Multi-Model Matrix - Kaggle Notebook

**Hardware:** 2x NVIDIA T4 (16GB VRAM each)
**Tests:** 5 different distillation scenarios
**Runtime:** ~15-25 minutes total

## Test Matrix
| # | Model Pair | Method | Dataset |
|---|------------|--------|---------|
| 1 | DistilGPT2 → DistilGPT2 | feature_distillation | wikitext-2 |
| 2 | gpt2-medium → gpt2 | feature_distillation | code_search_net |
| 3 | gpt2-medium → gpt2 | similarity_transfer | code_search_net |
| 4 | gpt2-medium → gpt2 | attention_transfer | code_search_net |
| 5 | gpt2-medium → gpt2 | multi_stage | code_search_net |

In [ ]:
# Check GPU setup
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  CUDA:{i} - {torch.cuda.get_device_name(i)}')
    print(f'    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate
!pip install -q pyyaml omegaconf rich
!pip install -q matplotlib seaborn tqdm lion-pytorch

## Clone Repository

In [ ]:
# REPLACE with your GitHub username and branch
from pathlib import Path
GITHUB_USERNAME = "YOUR_USERNAME"  # <-- CHANGE THIS
BRANCH = "main"  # or your feature branch
REPO_DIR = Path("/kaggle/working/zynthe")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{GITHUB_USERNAME}/zynthe.git {REPO_DIR}
%cd /kaggle/working/zynthe
print(f'cwd: {Path.cwd()}')
print(f'app/main.py exists: {Path("app/main.py").exists()}')

## Setup Directories & Helpers

In [ ]:
import os
import json
from pathlib import Path

# Create experiment output directory
os.makedirs("experiments", exist_ok=True)

EXPERIMENT_LOG = []

def run_experiment(name, config_file, overrides=None):
    '''Run distillation and log results'''
    import subprocess
    repo_dir = Path('/kaggle/working/zynthe')
    assert (repo_dir / 'app/main.py').exists(), 'Expected /kaggle/working/zynthe/app/main.py'
    
    cmd = ["python", "app/main.py", "distill", "--config", config_file]
    if not any(str(item).startswith("train.optimizer=") for item in (overrides or [])):
        cmd.extend(["--override", "train.optimizer=lion"])
    if overrides:
        for o in overrides:
            cmd.extend(["--override", o])
    
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print(f"{'='*60}")
    
    result = subprocess.run(cmd, cwd=str(repo_dir), capture_output=True, text=True)
    
    # Parse output for experiment ID
    exp_id = None
    for line in result.stdout.split('\n'):
        if 'Experiment:' in line:
            exp_id = line.split('Experiment:')[-1].strip()
    
    EXPERIMENT_LOG.append({
        "name": name,
        "config": config_file,
        "overrides": overrides,
        "experiment_id": exp_id,
        "success": result.returncode == 0
    })
    
    return result, exp_id

print("Helper functions ready")

## Test 1: DistilGPT2 Self-Distillation (wikitext-2)

In [ ]:
# Test 1: DistilGPT2 → DistilGPT2 with feature_distillation on wikitext-2
result, exp_id = run_experiment(
    name="test1_distilgpt2_self",
    config_file="configs/kaggle_quick_test.yaml",
    overrides=[
        "train.epochs=1",
        "data.train_samples=500"
    ]
)
print(result.stdout[-1000:] if result.stdout else "No output")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Test 2: GPT-2-medium → GPT-2 (feature_distillation)

In [ ]:
# Test 2: gpt2-medium → gpt2 with feature_distillation
result, exp_id = run_experiment(
    name="test2_gpt2_feature",
    config_file="configs/kaggle_dual_t4.yaml",
    overrides=[
        "train.epochs=1",
        "data.train_samples=500",
        "distillation.method=feature_distillation"
    ]
)
print(result.stdout[-1000:] if result.stdout else "No output")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Test 3: GPT-2-medium → GPT-2 (similarity_transfer)

In [ ]:
# Test 3: gpt2-medium → gpt2 with similarity_transfer
result, exp_id = run_experiment(
    name="test3_gpt2_similarity",
    config_file="configs/kaggle_dual_t4.yaml",
    overrides=[
        "train.epochs=1",
        "data.train_samples=500",
        "distillation.method=similarity_transfer"
    ]
)
print(result.stdout[-1000:] if result.stdout else "No output")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Test 4: GPT-2-medium → GPT-2 (attention_transfer)

In [ ]:
# Test 4: gpt2-medium → gpt2 with attention_transfer
result, exp_id = run_experiment(
    name="test4_gpt2_attention",
    config_file="configs/kaggle_dual_t4.yaml",
    overrides=[
        "train.epochs=1",
        "data.train_samples=500",
        "distillation.method=attention_transfer"
    ]
)
print(result.stdout[-1000:] if result.stdout else "No output")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Test 5: GPT-2-medium → GPT-2 (multi_stage)

In [ ]:
# Test 5: gpt2-medium → gpt2 with multi_stage (all losses)
result, exp_id = run_experiment(
    name="test5_gpt2_multistage",
    config_file="configs/kaggle_dual_t4.yaml",
    overrides=[
        "train.epochs=1",
        "data.train_samples=500",
        "distillation.method=multi_stage"
    ]
)
print(result.stdout[-1000:] if result.stdout else "No output")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Experiment Summary

In [ ]:
# Print experiment summary
print("\n")
print("="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
for exp in EXPERIMENT_LOG:
    status = "✅" if exp["success"] else "❌"
    print(f"{status} {exp['name']}")
    print(f"   Config: {exp['config']}")
    print(f"   ExpID:  {exp.get('experiment_id', 'N/A')}")

# Save log
with open("experiments/kaggle_matrix_log.json", "w") as f:
    json.dump(EXPERIMENT_LOG, f, indent=2)
print("\nLogged to experiments/kaggle_matrix_log.json")

---

# Commands for Inference, Visualization & Evaluation

## Inference Command

In [ ]:
# Inference - single text
!python app/main.py evaluate \
  --config configs/kaggle_dual_t4.yaml \
  --load-model-dir experiments/<EXPERIMENT_ID>/student_model \
  --infer-text "def fib(n): return fib(n-1) + fib(n-2)"

In [ ]:
# Inference - batch from file (one input per line)
!python app/main.py evaluate \
  --config configs/kaggle_dual_t4.yaml \
  --load-model-dir experiments/<EXPERIMENT_ID>/student_model \
  --infer-file my_test_inputs.txt \
  --infer-output my_predictions.json

## Visualization Commands

In [ ]:
# Plot training curves from training_metrics.json
import json
import matplotlib.pyplot as plt

metrics_path = "experiments/<EXPERIMENT_ID>/training_metrics.json"
if Path(metrics_path).exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss curve
    if 'train_loss' in metrics:
        axes[0].plot(metrics['train_loss'], label='train')
    if 'val_loss' in metrics:
        axes[0].plot(metrics['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].set_xlabel('Step')
    
    # Distillation loss components
    dist_keys = [k for k in metrics.keys() if 'distill' in k.lower() or 'kd' in k.lower()]
    for k in dist_keys:
        if isinstance(metrics[k], list):
            axes[1].plot(metrics[k], label=k)
    axes[1].set_title('Distillation Losses')
    axes[1].legend()
    axes[1].set_xlabel('Step')
    
    plt.tight_layout()
    plt.savefig("experiments/<EXPERIMENT_ID>/training_curves.png", dpi=150)
    plt.show()
    print("Saved to experiments/<EXPERIMENT_ID>/training_curves.png")
else:
    print("No training_metrics.json found")

## Evaluation Commands

In [ ]:
# Evaluate on validation set (uses config's data.val_samples)
!python app/main.py evaluate \
  --config configs/kaggle_dual_t4.yaml \
  --load-model-dir experiments/<EXPERIMENT_ID>/student_model

In [ ]:
# Compare teacher vs student
!python app/main.py compare \
  experiments/<TEACHER_EXP_ID>/student_model \
  experiments/<STUDENT_EXP_ID>/student_model \
  --output-dir experiments/comparison_results

---

## Logging & Persisted Artifacts

In [ ]:
# What gets saved per experiment:
print("""
experiments/<EXPERIMENT_ID>/
├── config.yaml           # Full config used
├── training_metrics.json # Loss curves, KD losses
├── student_model/        # Trained student (HF format)
│   ├── config.json
│   ├── model.safetensors
│   ├── tokenizer.json
│   └── ...
└── checkpoints/         # (if train.save_checkpoints=true)
    └── trainer_state.pt

# Additional logs:
experiments/kaggle_matrix_log.json  # All runs summary
<EXPERIMENT_ID>/training_curves.png   # Visualization
<INFERENCE_OUTPUT>.json              # Inference results
""")

## GPU Memory Check

In [ ]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    print(f'GPU {i}: Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB')